In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping


print("Tensorflow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


Tensorflow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
df = pd.read_csv('/content/ecommerce_complaints_250k_unique.csv')

df.rename(columns={'complaint_text': 'complaint'}, inplace=True)
print('Dataset shape', df.shape)
print('\n columns:')
print(df.columns.tolist())

df.head()


Dataset shape (250000, 3)

 columns:
['complaint_id', 'complaint', 'department']


,complaint_id,complaint,department
0,C038684,An unknown device accessed my account and I ca...,Account & Security
1,C064940,My support request remains unanswered after se...,Customer Support
2,C003955,My registered email was changed despite upload...,Account & Security
3,C120375,My shipment shows a delivery exception while t...,Order & Delivery
4,C172862,The product quality is not advertised after I ...,Product & Seller


In [3]:
#Basic validation

print("missing values\n")
print(df.isnull().sum())
print('\n Duplicate',df.duplicated().sum())
print('\n Deparment Distribution:\n')
print(df['department'].value_counts())

missing values

complaint_id    0
complaint       0
department      0
dtype: int64

 Duplicate 0

 Deparment Distribution:

department
Account & Security       41667
Customer Support         41667
Order & Delivery         41667
Payments & Refunds       41667
Product & Seller         41666
Returns & Replacement    41666
Name: count, dtype: int64


In [4]:
print("\nUnique departments:")
print(df["department"].unique())


Unique departments:
['Account & Security' 'Customer Support' 'Order & Delivery'
 'Product & Seller' 'Payments & Refunds' 'Returns & Replacement']


In [5]:
#Text Cleaning
df["complaint"] = (
    df["complaint"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["complaint"].head()

,complaint
0,an unknown device accessed my account and i ca...
1,my support request remains unanswered after se...
2,my registered email was changed despite upload...
3,my shipment shows a delivery exception while t...
4,the product quality is not advertised after i ...


In [6]:
#NLP Preprocessing

X = df['complaint']
y = df['department']

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExample complaint:")
print(X.iloc[0])

print("\nExample label:")
print(y.iloc[0])

print('\nMaxLen of Complaint')
print(df['complaint'].str.len().max())

X shape: (250000,)
y shape: (250000,)

Example complaint:
an unknown device accessed my account and i cannot reverse it, and the available instructions do not help. please restore my secure access, because the automated recovery process failed.

Example label:
Account & Security

MaxLen of Complaint
241


In [7]:
#Label Encoding

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print('Department mapping:')

for index, deparment in enumerate(label_encoder.classes_):
  print(f'{index} -> {deparment}')

Department mapping:
0 -> Account & Security
1 -> Customer Support
2 -> Order & Delivery
3 -> Payments & Refunds
4 -> Product & Seller
5 -> Returns & Replacement


In [8]:
#Train test Split

# First split: 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# Second split: 10% validation, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print(len(X_train), len(y_train))
print(len(X_val), len(y_val))
print(len(X_test), len(y_test))

Training: 200000
Validation: 25000
Testing: 25000
200000 200000
25000 25000
25000 25000


In [9]:
#Verify class distribution

print("Training distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation distribution:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Training distribution:
0    33333
1    33334
2    33334
3    33333
4    33333
5    33333
Name: count, dtype: int64

Validation distribution:
0    4167
1    4166
2    4167
3    4167
4    4166
5    4167
Name: count, dtype: int64

Test distribution:
0    4167
1    4167
2    4166
3    4167
4    4167
5    4166
Name: count, dtype: int64


In [10]:
!pip install chromadb

In [11]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print('model loaded success')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model loaded success


In [12]:
chroma_client = chromadb.PersistentClient(
    path = "/content/chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name = 'flatkart_complints'
)

print('Chromadb collection created.')

Chromadb collection created.


In [13]:
train_documents = X_train.tolist()

train_embeddings = embedding_model.encode(
  train_documents,
  batch_size = 128,
  show_progress_bar = True
)

print("Embedding generated")
print("shape:", train_embeddings.shape)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Embedding generated
shape: (200000, 384)


In [17]:
from tqdm.auto import tqdm

train_ids = [
    f"complaint_{i}"
    for i in range(len(train_documents))
]

train_metadatas = [
    {"label": int(label)}
    for label in y_train
]

BATCH_SIZE = 5000

for start in range(0, len(train_documents), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(train_documents))

    collection.add(
        ids=train_ids[start:end],
        documents=train_documents[start:end],
        embeddings=train_embeddings[start:end].tolist(),
        metadatas=train_metadatas[start:end]
    )

print("Embeddings added to ChromaDB")
print("Total vectors:", collection.count())

Embeddings added to ChromaDB
Total vectors: 200000


In [19]:
query = "Money was deducted but my payment did not complete."

query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings = [query_embedding.tolist()],
    n_results = 5
)

for i in range(5):

    label = results["metadatas"][0][i]["label"]

    department = label_encoder.inverse_transform(
        [label]
    )[0]

    print("\nComplaint:")
    print(results["documents"][0][i])

    print("Department:")
    print(department)

    print("Distance:")
    print(results["distances"][0][i])


Complaint:
money was deducted without order confirmation after the payment attempt failed. please correct the payment status because while the expected processing time has passed, because this charge remains unresolved.
Department:
Payments & Refunds
Distance:
0.3994155526161194

Complaint:
i was charged the wrong amount. even though money was deducted and i cannot confirm where money went. please explain the payment failure, because i checked my bank statement.
Department:
Payments & Refunds
Distance:
0.4137868583202362

Complaint:
money was deducted without order confirmation after the payment attempt failed while the expected processing time has passed. please review the transaction history. because this charge remains unresolved.
Department:
Payments & Refunds
Distance:
0.42093947529792786

Complaint:
money was deducted without order confirmation. on every payment attempt while the expected processing time has passed. please correct the payment status, because this charge remains 

In [21]:
from transformers import AutoTokenizer, AutoModel

T_name = 'sentence-transformers/all-MiniLM-L6-v2'

transformer_tokenizer = AutoTokenizer.from_pretrained(
    T_name
    )

transformer_model  = AutoModel.from_pretrained(
    T_name
)

print('Transformer model loaded')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Transformer model loaded


In [24]:
MAX_LEN = 64

print("MAX_LEN:", MAX_LEN)

MAX_LEN: 64


In [25]:
sample_text = X_train.iloc[0]

tokens = transformer_tokenizer(
    sample_text,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="tf"
)

outputs = transformer_model(
    input_ids=tokens["input_ids"],
    attention_mask=tokens["attention_mask"]
)

sample_token_embeddings = outputs.last_hidden_state

print("Token embedding shape:")
print(sample_token_embeddings.shape)

ValueError: tf is not a valid TensorType, please select one of ['pt', 'np', 'mlx']

In [ ]:
#Tokenizetion

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1


print("Vocabulary size:", vocab_size)

Vocabulary size: 425


In [ ]:
#Vocab inspection
print("First 20 vocabulary words:")

for word, index in list(tokenizer.word_index.items())[:20]:
    print(index, "->", word)
print("\nOOV token index:")
print(tokenizer.word_index.get("<OOV>"))

First 20 vocabulary words:
1 -> <OOV>
2 -> the
3 -> i
4 -> please
5 -> my
6 -> and
7 -> because
8 -> was
9 -> a
10 -> after
11 -> delivery
12 -> is
13 -> has
14 -> need
15 -> return
16 -> cannot
17 -> account
18 -> so
19 -> can
20 -> despite

OOV token index:
1


In [ ]:
#Convert Text into Sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print('Orignal:')
print(X_train.iloc[0])

print('\nSequence:')
print(X_train_seq[0])

print(tokenizer.word_index)

Orignal:
the received item differs from the photos after i checked the stated specifications. and i cannot trust the listing. please review the seller's representation. because i saved the original listing.

Sequence:
[2, 88, 26, 203, 82, 2, 411, 10, 3, 93, 2, 69, 107, 6, 3, 16, 218, 2, 29, 4, 32, 2, 71, 83, 7, 3, 226, 2, 47, 29]
{'<OOV>': 1, 'the': 2, 'i': 3, 'please': 4, 'my': 5, 'and': 6, 'because': 7, 'was': 8, 'a': 9, 'after': 10, 'delivery': 11, 'is': 12, 'has': 13, 'need': 14, 'return': 15, 'cannot': 16, 'account': 17, 'so': 18, 'can': 19, 'despite': 20, 'product': 21, 'remains': 22, 'without': 23, 'support': 24, 'not': 25, 'item': 26, 'process': 27, 'while': 28, 'listing': 29, 'which': 30, 'payment': 31, 'review': 32, 'information': 33, 'status': 34, 'provide': 35, 'order': 36, 'details': 37, 'leaving': 38, 'to': 39, 'correct': 40, 'seller': 41, 'replacement': 42, 'package': 43, 'investigate': 44, 'unresolved': 45, 'this': 46, 'original': 47, 'me': 48, 'transaction': 49, 'case'

In [ ]:
#Determine sequence length
train_lengths = [len(sequence) for sequence in X_train_seq]

print("Minimum:", np.min(train_lengths))
print("Maximum:", np.max(train_lengths))
print("Mean:", np.mean(train_lengths))
print("95th percentile:", np.percentile(train_lengths, 95))
print("99th percentile:", np.percentile(train_lengths, 99))

MAX_LEN = int(np.percentile(train_lengths, 95))

print("MAX_LEN:", MAX_LEN)


Minimum: 24
Maximum: 30
Mean: 27.64318
95th percentile: 30.0
99th percentile: 30.0
MAX_LEN: 30


In [ ]:
#Padding

# X = df['complaint']
# y = pd.get_dummies(df['department']).values

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

print("Training shape:",X_train_pad.shape)
print("X_val_pad:", X_val_pad.shape)
print("Testing shape:",X_test_pad.shape)

Training shape: (200000, 30)
X_val_pad: (25000, 30)
Testing shape: (25000, 30)


In [ ]:
#Example of padding
print("Sequence:")
print(X_train_seq[5])

print("\nPadded:")
print(X_train_pad[5])


Sequence:
[373, 374, 62, 375, 5, 171, 10, 204, 196, 114, 349, 4, 35, 165, 254, 181, 7, 30, 133, 2, 27, 299, 300, 3, 14, 261, 39, 146, 161]

Padded:
[373 374  62 375   5 171  10 204 196 114 349   4  35 165 254 181   7  30
 133   2  27 299 300   3  14 261  39 146 161   0]


In [23]:
#Model building

model = Sequential()

model.add(
    LSTM(
        64,
        dropout = 0.2,
        rerurn_sequences = True,
        input_shape = (MAX_LEN, 384)
    )
)


model.add(LSTM(32))
model.add(LSTM(16))

model.add(Dense(len(label_encoder.classes_), activation='softmax'))

model.summary()

NameError: name 'MAX_LEN' is not defined

In [ ]:
#Compile Model

model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)


In [ ]:
print("X_train_pad:", X_train_pad.shape)
print("y_train:", y_train.shape)

print("X_val_pad:", X_val_pad.shape)
print("y_val:", y_val.shape)

print("X_test_pad:", X_test_pad.shape)
print("y_test:", y_test.shape)

X_train_pad: (200000, 30)
y_train: (200000,)
X_val_pad: (25000, 30)
y_val: (25000,)
X_test_pad: (25000, 30)
y_test: (25000,)


In [ ]:
#Model Training

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    min_delta=0.1,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data = (X_val_pad, y_val),
    epochs=10,
    batch_size = 60,
    callbacks = [early_stopping],
    verbose = 1
)

Epoch 1/10
3334/3334 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - accuracy: 0.9917 - loss: 0.0291 - val_accuracy: 1.0000 - val_loss: 1.1500e-04
Epoch 2/10
3334/3334 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - accuracy: 1.0000 - loss: 4.6946e-05 - val_accuracy: 1.0000 - val_loss: 1.2700e-05
Epoch 3/10
3334/3334 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - accuracy: 1.0000 - loss: 6.0496e-06 - val_accuracy: 1.0000 - val_loss: 1.9180e-06
Epoch 4/10
3334/3334 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - accuracy: 1.0000 - loss: 9.4339e-07 - val_accuracy: 1.0000 - val_loss: 3.0337e-07


In [ ]:
#Evaluation
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 1.0000 - loss: 1.1420e-04

Test Loss: 0.00011419707152526826
Test Accuracy: 1.0


In [ ]:
from re import VERBOSE
from typing import Sequence
# Testing on unseen data

def predict_dept(complaint):

  complaint = (str(complaint).strip().lower())

  sequence = tokenizer.texts_to_sequences([complaint])


  padded = pad_sequences(
      sequence,
      maxlen = MAX_LEN,
      padding = 'post',
      truncating = 'post'
  )


  probs = model.predict(
      padded,
      verbose = 0
  )[0]

  predicted_index = np.argmax(probs)

  predicted_dept = label_encoder.inverse_transform([predicted_index])[0]

  confidence = probs[predicted_index]

  return predicted_dept, confidence




In [ ]:
complaint = "I received the product but I want to send it back and get my money refunded."

deparment, confidence = predict_dept(complaint)

print('Complaint:', complaint)
print("Predicted Department:", deparment)
print("Confidence:", confidence)


Complaint: I received the product but I want to send it back and get my money refunded.
Predicted Department: Returns & Replacement
Confidence: 0.9886678


In [ ]:
hard_unseen_complaints = [
    "I have been locked out of my profile since this morning and none of my credentials work.",
    "My bank balance went down but the transaction on the website has not completed.",
    "It has been several days and the parcel is still nowhere to be found.",
    "What arrived at my house is not the item I selected when placing the order.",
    "The item I purchased is faulty and I need another one.",
    "I no longer need this purchase and would like to send it back."
]

for complaint in hard_unseen_complaints:

    department, confidence = predict_dept(complaint)

    print("\nComplaint:", complaint)
    print("Predicted:", department)
    print("Confidence:", round(confidence * 100, 2), "%")


Complaint: I have been locked out of my profile since this morning and none of my credentials work.
Predicted: Account & Security
Confidence: 100.0 %

Complaint: My bank balance went down but the transaction on the website has not completed.
Predicted: Payments & Refunds
Confidence: 99.98 %

Complaint: It has been several days and the parcel is still nowhere to be found.
Predicted: Customer Support
Confidence: 99.98 %

Complaint: What arrived at my house is not the item I selected when placing the order.
Predicted: Order & Delivery
Confidence: 58.37 %

Complaint: The item I purchased is faulty and I need another one.
Predicted: Returns & Replacement
Confidence: 98.65 %

Complaint: I no longer need this purchase and would like to send it back.
Predicted: Payments & Refunds
Confidence: 99.98 %


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/Complaint_Redirector/Models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory:", MODEL_DIR)

Model directory: /content/drive/MyDrive/Complaint_Redirector/Models


In [ ]:
MODEL_PATH = f"{MODEL_DIR}/complaint_lstm.keras"

model.save(MODEL_PATH)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
/content/drive/MyDrive/Complaint_Redirector/Models/complaint_lstm.keras


In [ ]:
import pickle

TOKENIZER_PATH = f"{MODEL_DIR}/tokenizer.pkl"

with open(TOKENIZER_PATH, "wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved successfully:")
print(TOKENIZER_PATH)

Tokenizer saved successfully:
/content/drive/MyDrive/Complaint_Redirector/Models/tokenizer.pkl


In [ ]:
LABEL_ENCODER_PATH = f"{MODEL_DIR}/label_encoder.pkl"

with open(LABEL_ENCODER_PATH, "wb") as file:
    pickle.dump(label_encoder, file)

print("Label encoder saved successfully:")
print(LABEL_ENCODER_PATH)

Label encoder saved successfully:
/content/drive/MyDrive/Complaint_Redirector/Models/label_encoder.pkl


In [ ]:
import os

print("Model exists:",
      os.path.exists(MODEL_PATH))

print("Tokenizer exists:",
      os.path.exists(TOKENIZER_PATH))

print("Label encoder exists:",
      os.path.exists(LABEL_ENCODER_PATH))

Model exists: True
Tokenizer exists: True
Label encoder exists: True


In [ ]:
for file_path in [
    MODEL_PATH,
    TOKENIZER_PATH,
    LABEL_ENCODER_PATH
]:
    size_mb = os.path.getsize(file_path) / (1024 * 1024)

    print(
        os.path.basename(file_path),
        "->",
        round(size_mb, 2),
        "MB"
    )

complaint_lstm.keras -> 0.72 MB
tokenizer.pkl -> 0.01 MB
label_encoder.pkl -> 0.0 MB


In [ ]:
from tensorflow.keras.models import load_model
import pickle

loaded_model = load_model(MODEL_PATH)

with open(TOKENIZER_PATH, "rb") as file:
    loaded_tokenizer = pickle.load(file)

with open(LABEL_ENCODER_PATH, "rb") as file:
    loaded_label_encoder = pickle.load(file)

print("All artifacts loaded successfully.")
loaded_model.summary()

print("Vocabulary size:",
      len(loaded_tokenizer.word_index) + 1)

print("Departments:")

for index, department in enumerate(
    loaded_label_encoder.classes_
):
    print(index, "->", department)

All artifacts loaded successfully.


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 30, 100)        │        42,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 32)             │        17,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,464 (704.94 KB)

 Trainable params: 60,154 (234.98 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 120,310 (469.96 KB)

Vocabulary size: 425
Departments:
0 -> Account & Security
1 -> Customer Support
2 -> Order & Delivery
3 -> Payments & Refunds
4 -> Product & Seller
5 -> Returns & Replacement


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

complaint = (
    "I was charged for my order but the payment "
    "is still showing as pending."
)

# Same preprocessing
complaint = (
    complaint
    .strip()
    .lower()
)

# Use SAVED tokenizer
sequence = loaded_tokenizer.texts_to_sequences(
    [complaint]
)

# Use the same MAX_LEN from training
padded = pad_sequences(
    sequence,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

# Use SAVED model
probabilities = loaded_model.predict(
    padded,
    verbose=0
)[0]

predicted_index = np.argmax(probabilities)

# Use SAVED label encoder
predicted_department = (
    loaded_label_encoder.inverse_transform(
        [predicted_index]
    )[0]
)

confidence = probabilities[predicted_index]

print("Complaint:", complaint)
print("Predicted Department:", predicted_department)
print("Confidence:", round(confidence * 100, 2), "%")

Complaint: i was charged for my order but the payment is still showing as pending.
Predicted Department: Payments & Refunds
Confidence: 97.71 %


In [ ]:
from google.colab import files

files.download(MODEL_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download(TOKENIZER_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download(LABEL_ENCODER_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>